# E1 — Label Smoothing (IndoBERTweet-LoRA)

**Eksperimen E1** dari plan E1-E3 (docs/LOG_EKSPERIMEN.md). Baseline E0 (run v8, kolom `text_bert`):
acc 0.7954 / Macro F1 0.7328 / netral F1 0.55.

**Desain**: konfigurasi terbaik trial-4 (batch 16, dropout 0.3, lr 2e-4, r 16, alpha 32),
sweep `label_smoothing_factor` eps = {0.0, 0.05, 0.10, 0.15}, 5 epoch,
`load_best_model_at_end` per val macro F1. Evaluasi test untuk baseline vs eps terbaik,
simpan prediksi + probabilitas, uji McNemar.

**Ekspektasi**: Macro F1 +1-3 poin, recall netral naik, precision negatif sedikit turun.


In [ ]:
# P0.1 (replikasi): T4 x2 membuat Trainer memakai DataParallel -> batch efektif 32
# (bukan 16) dan jumlah update separuh (975 vs 1950 step) -> undertrained
# (loss ~1.35 vs 0.64 kanonik) + crash WeightedTrainer ('DataParallel' has no
# attribute 'config'). Paksa 1 GPU agar protokol identik dengan run kanonik.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))

In [ ]:
# torchao 0.10 di image Kaggle tidak kompatibel dengan peft (butuh > 0.16).
# Wajib di-uninstall sebelum build model (lihat docs/PLAN_PASCA_RERUN.md).
!pip uninstall -y torchao

In [ ]:
# P0.3 (replikasi): image Kaggle membawa transformers 5.0.0 (mayor), run kanonik era 4.x.
# E1 v4 (stack 5.0): val macro F1 0.655 vs kanonik 0.723 pada data yang sama.
# --force-reinstall --no-deps + dependensi eksplisit: uninstall biasa ternyata
# meninggalkan file 5.0 (v5/v6: ImportError TFPreTrainedModel dari file campur),
# dan --no-deps menghindari resolver gagal karena dependensi image terlalu baru.
!pip install --force-reinstall --no-deps "transformers==4.46.3" "peft==0.13.2" "tokenizers==0.20.3" "huggingface-hub==0.26.5"

In [ ]:
# Sel identitas run - WAJIB untuk replikasi. Sengaja SETELAH pin versi (P0.3):
# import transformers di sini memakai stack yang SUDAH dipin, dan guard di bawah
# gagal keras bila instalasi masih campur (kasus v5/v6).
import sys
import torch
import transformers
import peft

assert transformers.__version__.startswith("4.46"), (
    f"transformers {transformers.__version__} bukan pin 4.46 - instalasi bermasalah!"
)
from transformers import TFPreTrainedModel  # bukti tidak ada file campur 5.0

print("python        :", sys.version)
print("torch         :", torch.__version__)
print("transformers  :", transformers.__version__)
print("peft          :", peft.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu count     :", torch.cuda.device_count())
print("gpu name      :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# =====================================================
# IMPORT LIBRARY
# =====================================================
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
# =====================================================
# SET SEED
# =====================================================
seed = 42
set_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print("GPU tersedia:", torch.cuda.is_available())

In [ ]:
# =====================================================
# LOAD DATA + SPLIT 80:20 (kolom BERT EKSPLISIT: text_bert)
# =====================================================
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Diagnostik: apa yang benar-benar ter-mount di /kaggle/input
print("Isi /kaggle/input:")
mounted = []
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        p = os.path.join(root, f)
        mounted.append(p)
        print("  ", p)
if not mounted:
    print("  (KOSONG - dataset tidak ter-mount!)")

DATASET_DIR = "/kaggle/input/thesis-indobert-processed-data"
# P0.2: path mount berbeda antar skema - CLI 2.x memakai
# /kaggle/input/datasets/<owner>/<slug>/, kernel lama (UI) memakai /kaggle/input/<slug>/.
# Ambil path CSV langsung dari daftar mount agar bekerja di keduanya.
csv_path = next(
    (p for p in mounted if p.endswith("data_preprocessed_with_emoticon.csv")), None
)
if csv_path is None:
    raise FileNotFoundError(
        "CSV dataset tidak ditemukan di mount. Ter-mount: "
        + (str(mounted) if mounted else "TIDAK ADA APA PUN")
        + " - cek dataset_sources di kernel-metadata.json / attach manual via UI."
    )
print("CSV ditemukan di:", csv_path)
df = pd.read_csv(csv_path)

# P0: kolom BERT eksplisit (text_bert) - tidak ada fallback diam-diam
if "text_bert" in df.columns:
    col_bert = "text_bert"
else:
    raise ValueError(
        "Kolom 'text_bert' tidak ditemukan di CSV. Kolom tersedia: " + str(df.columns.tolist())
    )
col_label = "label"
print("Kolom BERT terpilih:", col_bert)

df[col_bert] = df[col_bert].fillna("").astype(str)

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df[col_label]
)

X_train_bert = train_df[col_bert].values
X_test_bert = test_df[col_bert].values
y_train = train_df[col_label].values
y_test = test_df[col_label].values

print(f"Train set: {len(X_train_bert)} | Test set: {len(X_test_bert)}")

In [ ]:
# =====================================================
# SPLIT TRAIN -> TRAIN FINAL + VALIDATION (10%)
# =====================================================
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_bert, y_train, test_size=0.1, stratify=y_train, random_state=42
)

print("Distribusi train final:")
print(pd.Series(y_train_final).value_counts().sort_index())
print()
print("Distribusi validation:")
print(pd.Series(y_val).value_counts().sort_index())
print()
print("Distribusi test:")
print(pd.Series(y_test).value_counts().sort_index())

In [ ]:
# =====================================================
# TOKENIZER
# =====================================================
model_name = "indolem/indobertweet-base-uncased"
tokenizer_bert = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# =====================================================
# DATASET PYTORCH (versi numpy-friendly)
# =====================================================
import torch
from torch.utils.data import Dataset
import numpy as np
import pandas as pd

class SentimenDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.values if isinstance(texts, pd.Series) else np.array(texts)
        self.labels = labels.values if isinstance(labels, pd.Series) else np.array(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }

In [ ]:
# =====================================================
# METRIK EVALUASI (average='macro', zero_division=0)
# =====================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

In [ ]:
# =====================================================
# FUNGSI BUILD INDOBERTWEET-LORA
# =====================================================
def build_indobertweet_lora(dropout=0.3, r=16, lora_alpha=32):
    id2label = {0: "negatif", 1: "netral", 2: "positif"}
    label2id = {"negatif": 0, "netral": 1, "positif": 2}

    config = AutoConfig.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id,
        hidden_dropout_prob=dropout,
        attention_probs_dropout_prob=dropout,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, config=config, ignore_mismatched_sizes=True
    )
    model.config.id2label = id2label
    model.config.label2id = label2id

    lora_config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        target_modules=["query", "value"],
        lora_dropout=dropout,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        modules_to_save=["classifier"],
    )
    return get_peft_model(model, lora_config)

In [ ]:
# =====================================================
# BUAT DATASET TRAIN/VAL/TEST
# =====================================================
train_dataset = SentimenDataset(X_train_final, y_train_final, tokenizer_bert, max_length=128)
val_dataset = SentimenDataset(X_val, y_val, tokenizer_bert, max_length=128)
test_dataset = SentimenDataset(X_test_bert, y_test, tokenizer_bert, max_length=128)

print(f"Train {len(train_dataset)} | Val {len(val_dataset)} | Test {len(test_dataset)}")

## E1 - Sweep Label Smoothing

4 pelatihan: eps = 0 (baseline), 0.05, 0.10, 0.15. Semua diukur di validation;
sanity check collapse per eps. Model ditahan di memori untuk evaluasi test.

In [ ]:
# =====================================================
# E1 - SWEEP LABEL SMOOTHING
# =====================================================
EPS_GRID = [0.0, 0.05, 0.10, 0.15]  # sweep penuh di label BARU (corrected)
best_params = {"batch_size": 16, "dropout": 0.3, "learning_rate": 0.0002, "r": 16, "alpha": 32}

e1_trainers = {}
e1_val_results = []

for eps in EPS_GRID:
    tag = "baseline" if eps == 0 else "eps" + str(int(round(eps * 100))).zfill(2)
    print()
    print("=" * 60)
    print(f"E1 | label_smoothing_factor = {eps} | tag = {tag}")
    print("=" * 60)

    set_seed(seed)
    model = build_indobertweet_lora(
        dropout=best_params["dropout"], r=best_params["r"], lora_alpha=best_params["alpha"]
    )

    training_args = TrainingArguments(
        output_dir=f"./results_e1_{tag}",
        learning_rate=best_params["learning_rate"],
        per_device_train_batch_size=best_params["batch_size"],
        per_device_eval_batch_size=best_params["batch_size"],
        num_train_epochs=5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=50,
        report_to="none",
        save_total_limit=1,
        label_smoothing_factor=eps,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    eval_result = trainer.evaluate()
    print("Hasil validation:", eval_result)

    # --- Sanity check (P0): deteksi collapse ---
    preds_val = trainer.predict(val_dataset)
    y_val_pred = np.argmax(preds_val.predictions, axis=1)
    maj = pd.Series(y_val).mode()[0]
    p_maj = float((y_val == maj).mean())
    f1_maj = (2 * p_maj / (1 + p_maj)) / 3
    print("Distribusi prediksi val :", pd.Series(y_val_pred).value_counts().sort_index().to_dict())
    print(f"Baseline mayoritas val  : acc={p_maj:.4f} macro_f1={f1_maj:.4f}")
    status = "COLLAPSE" if eval_result["eval_f1_macro"] <= f1_maj + 1e-6 else "OK"
    print("STATUS:", status)

    e1_trainers[tag] = trainer
    e1_val_results.append({
        "tag": tag,
        "label_smoothing": eps,
        "accuracy_val": eval_result["eval_accuracy"],
        "precision_macro_val": eval_result["eval_precision_macro"],
        "recall_macro_val": eval_result["eval_recall_macro"],
        "f1_macro_val": eval_result["eval_f1_macro"],
        "status": status,
    })

In [ ]:
# =====================================================
# RINGKASAN VALIDATION + PILIH EPS TERBAIK
# =====================================================
df_e1_val = pd.DataFrame(e1_val_results).sort_values("f1_macro_val", ascending=False)
print(df_e1_val.to_string(index=False))
best_tag = df_e1_val.iloc[0]["tag"]
print("Tag terbaik (val macro F1):", best_tag)
df_e1_val.to_csv("hasil_e1_val.csv", index=False)

In [ ]:
# =====================================================
# EVALUASI TEST: baseline vs eps terbaik (+ simpan probabilitas)
# =====================================================
def softmax_np(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

label_names = ["negatif", "netral", "positif"]
mapping = {0: "negatif", 1: "netral", 2: "positif"}
e1_test_results = []
e1_test_preds = {}

for tag in list(dict.fromkeys(["baseline", best_tag])):  # dedup, jaga urutan
    print()
    print("=" * 60)
    print("EVALUASI TEST |", tag)
    print("=" * 60)
    trainer = e1_trainers[tag]
    preds_test = trainer.predict(test_dataset)
    logits = preds_test.predictions
    y_pred_test = np.argmax(logits, axis=1)
    P = softmax_np(logits)
    e1_test_preds[tag] = y_pred_test

    print(classification_report(y_test, y_pred_test, target_names=label_names, zero_division=0))
    print("Distribusi prediksi:", pd.Series(y_pred_test).value_counts().sort_index().to_dict())
    print("Distribusi aktual  :", pd.Series(y_test).value_counts().sort_index().to_dict())

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_test, y_pred_test, average="macro", zero_division=0
    )
    acc = accuracy_score(y_test, y_pred_test)
    e1_test_results.append({
        "tag": tag,
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
    })

    hasil = pd.DataFrame({
        "text": pd.Series(X_test_bert),
        "label_aktual": pd.Series(y_test),
        "label_prediksi": pd.Series(y_pred_test),
        "sentimen_prediksi": pd.Series(y_pred_test).map(mapping),
        "prob_negatif": P[:, 0],
        "prob_netral": P[:, 1],
        "prob_positif": P[:, 2],
    })
    fname = f"hasil_e1_test_{tag}.csv"
    hasil.to_csv(fname, index=False)
    print("Tersimpan:", fname)

df_e1_test = pd.DataFrame(e1_test_results)
print(df_e1_test.to_string(index=False))
df_e1_test.to_csv("hasil_e1_test.csv", index=False)

In [ ]:
# =====================================================
# McNEMAR EXACT: baseline vs eps terbaik
# (implementasi sama dengan quality_pipeline/verify_metrics.py)
# =====================================================
import math

yt = np.asarray(y_test)
a_ok = e1_test_preds["baseline"] == yt
b_ok = e1_test_preds[best_tag] == yt
b_count = int(np.sum(a_ok & ~b_ok))
c_count = int(np.sum(~a_ok & b_ok))
n_disc = b_count + c_count
if n_disc == 0:
    p_value = 1.0
else:
    tail = sum(math.comb(n_disc, i) for i in range(max(b_count, c_count), n_disc + 1)) / (2 ** n_disc)
    p_value = min(1.0, 2 * tail)

print(f"b (baseline benar, best salah) = {b_count}")
print(f"c (baseline salah, best benar) = {c_count}")
verdict = "SIGNIFIKAN" if p_value < 0.05 else "tidak signifikan"
print(f"p-value = {p_value:.6f} ({verdict} @ 0.05)")

In [ ]:
# =====================================================
# RINGKASAN AKHIR E1
# =====================================================
print("=== RINGKASAN E1 (validation) ===")
print(df_e1_val.to_string(index=False))
print()
print("=== RINGKASAN E1 (test) ===")
print(df_e1_test.to_string(index=False))
print()
print(f"Tag terbaik: {best_tag}")
print("Entri PASCA eksperimen diisi di docs/LOG_EKSPERIMEN.md setelah run selesai.")